In [90]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [91]:
# PARTE 2 --------------------------------------------------------------------------------------------------------

In [92]:
df = pd.read_csv("ventas_ecommerce_limpio.csv")

In [93]:
#Muestra las primeras filas. 
df.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [94]:
#Revisa las columnas.
df.columns

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [95]:
#Muestra cuantas filas y columnas tiene.
df.shape

(60, 10)

In [96]:
#Revisa si hay valores nulos.
df.isnull().sum()

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [97]:
#Verifica que exista la columna total_venta.


In [98]:
#Verifica que total_venta coincida con cantidad * precio_unitario.
df["total_calculado"] = df["cantidad"] * df["precio_unitario"]
df[df["total_venta"] != df["total_calculado"]]

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta,total_calculado


In [99]:
#Escribe una observacion breve sobre el estado del dataset.
# Puedo observar que el dataset ya cumple con buenas prácticas como normalización en nombres de columnas y también en sus valores como el 
# metodo de pago o ciudad, los cuales comienzan con letra mayuscula, al igual que el tipado es correcto para los valores (precio unitario, total venta
# y cantidad son de tipo númerico). Así mismo no se incluyen nulos y la cantidad total de datos es considerablemente mejor con respecto a los 
# otros datasets que hemos estado trabajando


In [100]:
# PARTE 3 --------------------------------------------------------------------------------------------------------

In [101]:
#Crea la columna venta_alta
df["venta_alta"] = df["total_venta"].apply(lambda x: 1 if x >= 1000 else 0)

In [102]:
#Cuenta cuantas ventas quedaron como 1 y cuantas como 0.
df["venta_alta"].value_counts()

venta_alta
1    40
0    20
Name: count, dtype: int64

In [103]:
# Por que venta_alta es la variable objetivo? porque prácticamente es una medida útil para clasificar si son ventas altas o bajas, lo 
# cuál nos ayuda a predecir

In [104]:
# PARTE 4 --------------------------------------------------------------------------------------------------------

In [105]:
#Crea X.
X = df[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]

In [106]:
#Crea y.
y = df["venta_alta"]

In [107]:
#Convierte variables categoricas con pd.get_dummies().
X = pd.get_dummies(X)

In [108]:
#Guarda la lista de columnas generadas.
columnas_modelo = X.columns.tolist()

In [109]:
#Muestra las primeras filas de X despues de get_dummies().
X.head()

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,250.0,True,False,False,True,False,False,True,False,False,False
1,1,650.0,True,False,False,False,True,False,False,False,True,False
2,1,900.0,True,False,False,False,True,False,False,False,False,True
3,1,800.0,True,False,False,True,False,False,True,False,False,False
4,2,180.0,True,False,False,True,False,False,False,False,True,False


In [110]:
#Por que no se debe usar total_venta como variable de entrada si venta_alta se creo a partir de total_venta? porque ya está incluida en 
# nuestras variables de entrada 

In [111]:
# PARTE 5 --------------------------------------------------------------------------------------------------------

In [112]:
#Divide los datos en entrenamiento y prueba.
#Usa 80% entrenamiento y 20% prueba.
#Usa random_state=42.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [113]:
#Entrena el modelo.
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [114]:
#Genera predicciones con los datos de prueba.
predicciones = modelo.predict(X_test)

In [115]:
#Calcula exactitud.
exactitud = accuracy_score(y_test, predicciones)
print("Exactitud:", exactitud)

Exactitud: 1.0


In [116]:
#Muestra matriz de confusion.
matriz = confusion_matrix(y_test, predicciones)
print(matriz)

[[4 0]
 [0 8]]


In [117]:
#Crea una tabla llamada resultados_prueba con: valor_real prediccion coincide
resultados_prueba = pd.DataFrame({
    "valor_real": y_test,
    "prediccion": predicciones
})
resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]

In [118]:
#Cuenta cuantos aciertos y cuantos errores hubo.
resultados_prueba["coincide"].value_counts()

coincide
True    12
Name: count, dtype: int64

In [119]:
aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]

print("aciertos: ", aciertos)
print("errores: ", errores)

aciertos:      valor_real  prediccion  coincide
0            0           0      True
5            0           0      True
36           1           1      True
45           1           1      True
13           0           0      True
54           1           1      True
33           1           1      True
48           1           1      True
12           0           0      True
57           1           1      True
46           1           1      True
50           1           1      True
errores:  Empty DataFrame
Columns: [valor_real, prediccion, coincide]
Index: []


In [120]:
#Cual fue la exactitud? del 100%
#Cuantos aciertos tuvo el modelo? 12
#Cuantos errores tuvo el modelo? 0
#Que indica la matriz de confusion? indica que hubieron 8 y 4 correctos; 0 y 0 incorrectos
#Una buena exactitud significa que el modelo ya es perfecto? Explica. Sí y no a la vez yo considero, porque los datos no se van a quedar como están para siempre ahora mismo y a pesar de esto, lo que estamos haciendo es prediciendo algo que posiblemente después no será un hecho

In [121]:
# PARTE 6 --------------------------------------------------------------------------------------------------------

In [122]:
#Guarda el modelo entrenado.
joblib.dump(modelo, "modelo_examen_venta_alta.pkl")

['modelo_examen_venta_alta.pkl']

In [123]:
#Guarda la lista de columnas usadas durante el entrenamiento.
joblib.dump(columnas_modelo, "columnas_examen_modelo.pkl")

['columnas_examen_modelo.pkl']

In [124]:
#Para que sirve guardar el modelo? para hacer uso del mismo con nuevos datos futuros
#Para que sirve guardar las columnas del entrenamiento? para no tener problemas al volver a usarlo después con datos nuevos y no perder las columnas que originalmente se usaron
#Que problema puede aparecer si no guardas las columnas? que el modelo no las cargue, por consiguiente, no sería funcional

In [125]:
# PARTE 7 --------------------------------------------------------------------------------------------------------

In [126]:
nuevas = pd.DataFrame({
    "id_venta": [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010],
    "fecha": ["2026-08-07", "2026-07-31", "2026-07-23", "2026-07-14", "2026-07-07", "2026-08-03", "2026-08-05", "2026-05-25", "2026-06-07", "2026-03-07"],
    "cliente": ["Melanie", "Amador", "Liliana", "Antonio", "Rocío", "Blanca", "Karla", "Diego", "Raúl", "Juan José"],
    "producto": ["Mouse", "Adaptador tipo C", "Calcetines", "Mousepad", "Laptop", "TV 40'", "Monitor curvo", "CPU", "Teclado", "Audifonos"],
    "categoria": ["Accesorios", "Electronica", "Ropa", "Accesorios", "Electronica", "Electronica", "Electronica", "Electronica", "Accesorios", "Electronica"],
    "cantidad": [1, 1, 4, 1, 1, 1, 1, 1, 1, 1],
    "precio_unitario": [500, 200, 40, 50, 13000, 6000, 5000, 3500, 999, 1001],
    "metodo_pago": ["Efectivo", "Efectivo", "Efectivo", "Efectivo", "Tarjeta", "Tarjeta", "Tarjeta", "Tarjeta", "Efectivo", "Tarjeta"],
    "ciudad": ["Cuernavaca", "Jiutepec", "Cuernavaca", "Cuernavaca", "Jojutla", "Jiutepec", "Jojutla", "Jojutla", "Cuernavaca", "Cuernavaca"]
})

nuevas.to_csv("examen_ventas_nuevas.csv", index=False)

In [127]:
# PARTE 8 --------------------------------------------------------------------------------------------------------

In [128]:
#Carga modelo_examen_venta_alta.pkl.
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")

In [129]:
#Carga columnas_examen_modelo.pkl.
columnas_modelo = joblib.load("columnas_examen_modelo.pkl")

In [130]:
#Carga examen_ventas_nuevas.csv.
nuevas = pd.read_csv("examen_ventas_nuevas.csv")

In [131]:
#Selecciona las mismas variables de entrada usadas en entrenamiento.
X_nuevas = nuevas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]

In [132]:
#Aplica pd.get_dummies().
X_nuevas = pd.get_dummies(X_nuevas)

In [133]:
#Alinea columnas con reindex().
X_nuevas = X_nuevas.reindex(columns=columnas_modelo, fill_value=0)

In [134]:
#Genera predicciones.
#Agrega la columna: prediccion_venta_alta
nuevas["prediccion_venta_alta"] = modelo_cargado.predict(X_nuevas)
nuevas.head(10)

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta
0,2001,2026-08-07,Melanie,Mouse,Accesorios,1,500,Efectivo,Cuernavaca,0
1,2002,2026-07-31,Amador,Adaptador tipo C,Electronica,1,200,Efectivo,Jiutepec,1
2,2003,2026-07-23,Liliana,Calcetines,Ropa,4,40,Efectivo,Cuernavaca,1
3,2004,2026-07-14,Antonio,Mousepad,Accesorios,1,50,Efectivo,Cuernavaca,0
4,2005,2026-07-07,Rocío,Laptop,Electronica,1,13000,Tarjeta,Jojutla,1
5,2006,2026-08-03,Blanca,TV 40',Electronica,1,6000,Tarjeta,Jiutepec,1
6,2007,2026-08-05,Karla,Monitor curvo,Electronica,1,5000,Tarjeta,Jojutla,1
7,2008,2026-05-25,Diego,CPU,Electronica,1,3500,Tarjeta,Jojutla,1
8,2009,2026-06-07,Raúl,Teclado,Accesorios,1,999,Efectivo,Cuernavaca,0
9,2010,2026-03-07,Juan José,Audifonos,Electronica,1,1001,Tarjeta,Cuernavaca,1


In [135]:
#Agrega la columna: interpretacion_prediccion
nuevas["interpretacion_prediccion"] = nuevas["prediccion_venta_alta"].map({
    0: "Venta no alta",
    1: "Venta alta"
})

In [136]:
#Guarda el resultado como: examen_predicciones.csv
nuevas.to_csv("examen_predicciones.csv", index=False)

In [137]:
#Que podria pasar si no usas reindex antes de predecir? No se cargarían desde el principio los índices, basicamente lo que hace es como si hiciera un reconteo

In [138]:
# PARTE 9 --------------------------------------------------------------------------------------------------------

In [139]:
#Calcula total_estimado = cantidad * precio_unitario
nuevas["total_estimado"] = nuevas["cantidad"] * nuevas["precio_unitario"]

In [140]:
#Crea: venta_alta_real_estimada
nuevas["venta_alta_real_estimada"] = nuevas["total_estimado"].apply(lambda x: 1 if x >= 1000 else 0)

In [141]:
#Compara:prediccion_venta_alta Y venta_alta_real_estimada
#Crea la columna: coincide
nuevas["coincide"] = nuevas["prediccion_venta_alta"] == nuevas["venta_alta_real_estimada"]

In [142]:
#Cuenta cuantas predicciones coincidieron y cuantas no.
nuevas["coincide"].value_counts()

coincide
True     8
False    2
Name: count, dtype: int64

In [143]:
#Filtra las ventas que no coincidieron.
errores = nuevas[nuevas["coincide"] == False]

In [144]:
#Revisa si los errores estan cerca del limite de 1000.
nuevas["distancia_a_1000"] = (nuevas["total_estimado"] - 1000).abs()

In [145]:
cerca_limite = nuevas[nuevas["distancia_a_1000"] <= 200]
cerca_limite

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
8,2009,2026-06-07,Raúl,Teclado,Accesorios,1,999,Efectivo,Cuernavaca,0,Venta no alta,999,0,True,1
9,2010,2026-03-07,Juan José,Audifonos,Electronica,1,1001,Tarjeta,Cuernavaca,1,Venta alta,1001,1,True,1


In [146]:
#Guarda de nuevo examen_predicciones.csv con las columnas de auditoria.
nuevas.to_csv("examen_predicciones.csv", index=False)

In [147]:
#Cuantas ventas nuevas evaluaste? 10
#Cuantas fueron predichas como venta alta? 7
#Cuantas fueron predichas como venta no alta? 3
#Cuantas coincidieron con la regla manual? 8
#Cuantas no coincidieron? 2
#Que ventas no coincidieron? la 2002 y 2003
#Los errores estuvieron cerca del limite de 1000? no
#Que paso con la categoria nueva? nada
#Que paso con la ciudad nueva? nada